<a href="https://colab.research.google.com/github/alaaguedda/medical_report_summarization_project/blob/main/medical_summarization_project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ===============================
# Project setup and directories
# ===============================

import os

BASE_DIR = "/content/medical_summarization_project"

folders = [
    "data/raw",
    "data/ground_truth",
    "data/processed",
    "embeddings",
    "models/extractive",
    "models/abstractive",
    "results/extractive_summaries",
    "results/abstractive_summaries",
    "utils"
]

for folder in folders:
    os.makedirs(os.path.join(BASE_DIR, folder), exist_ok=True)

print("Project folders created.")


In [ ]:
# ===============================
# Core NLP libraries
# ===============================

import re
import json
import numpy as np
import pandas as pd

# NLP preprocessing
import nltk
from nltk.tokenize import sent_tokenize

# Vectorization & importance
from sklearn.feature_extraction.text import TfidfVectorizer

# Transformers
from transformers import (
    AutoTokenizer,
    AutoModel,
    AutoModelForSeq2SeqLM
)

# Similarity & evaluation
from sklearn.metrics.pairwise import cosine_similarity

# Download NLTK resources
nltk.download('punkt')


In [ ]:
# ===============================
# Load raw medical reports
# ===============================

def load_reports(path):
    reports = {}
    for file in sorted(os.listdir(path)):
        if file.endswith(".txt"):
            with open(os.path.join(path, file), "r") as f:
                reports[file] = f.read()
    return reports

raw_reports = load_reports(os.path.join(BASE_DIR, "data/raw"))
print(f"Loaded {len(raw_reports)} medical reports")


In [ ]:
# ===============================
# Text preprocessing
# ===============================

def preprocess_text(text):
    text = text.lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

processed_reports = {
    k: preprocess_text(v) for k, v in raw_reports.items()
}


In [ ]:
import nltk

nltk.download("punkt")
nltk.download("punkt_tab")


# ===============================
# Sentence segmentation
# ===============================

sentence_data = {}

for file, text in processed_reports.items():
    sentences = sent_tokenize(text)
    sentence_data[file] = sentences

    with open(
        os.path.join(BASE_DIR, "data/processed", file.replace(".txt", ".json")),
        "w"
    ) as f:
        json.dump(sentences, f, indent=2)

# ---- Inspection ----
print("Sentence segmentation complete.\n")

example_file = list(sentence_data.keys())[0]
example_sentences = sentence_data[example_file]

print(f"Example document: {example_file}")
print(f"Number of sentences: {len(example_sentences)}\n")

print("First 3 sentences:")
for i, s in enumerate(example_sentences[:3], 1):
    print(f"{i}. {s}")



In [ ]:
# ===============================
# TF-IDF importance
# ===============================

documents = list(processed_reports.values())

tfidf = TfidfVectorizer(
    ngram_range=(1, 2),
    max_features=5000
)

tfidf_matrix = tfidf.fit_transform(documents)
feature_names = tfidf.get_feature_names_out()

# ---- Inspection ----
print("TF-IDF matrix created.\n")
print("TF-IDF matrix shape:")
print(f"Documents: {tfidf_matrix.shape[0]}")
print(f"Vocabulary size: {tfidf_matrix.shape[1]}\n")

# Show top TF-IDF terms for one document
doc_id = 0
scores = tfidf_matrix[doc_id].toarray().flatten()
top_indices = scores.argsort()[-10:][::-1]

print("Top TF-IDF terms for first document:")
for idx in top_indices:
    print(f"{feature_names[idx]} → {scores[idx]:.4f}")


In [ ]:
# ===============================
# Transformer embeddings
# ===============================

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

def get_sentence_embedding(sentence):
    inputs = tokenizer(sentence, return_tensors="pt", truncation=True)
    outputs = model(**inputs)
    return outputs.last_hidden_state.mean(dim=1).detach().numpy()

sentence_embeddings = {}

for file, sentences in sentence_data.items():
    embeddings = [get_sentence_embedding(s)[0] for s in sentences]
    sentence_embeddings[file] = np.array(embeddings)

# ---- Inspection ----
example_file = list(sentence_embeddings.keys())[0]
emb = sentence_embeddings[example_file]

print("Sentence embeddings created.\n")
print(f"Example document: {example_file}")
print(f"Embedding matrix shape: {emb.shape}")
print("(num_sentences, embedding_dim)\n")

print("First sentence embedding (first 10 values):")
print(emb[0][:10])


In [ ]:
# ===============================
# Sentence importance scoring
# ===============================

def sentence_importance(embeddings):
    centroid = embeddings.mean(axis=0)
    scores = cosine_similarity(embeddings, centroid.reshape(1, -1))
    return scores.flatten()

# ---- Inspection ----
example_sentences = sentence_data[example_file]
example_embeddings = sentence_embeddings[example_file]

importance_scores = sentence_importance(example_embeddings)

print("Sentence importance scores computed.\n")
print(f"Scores shape: {importance_scores.shape}\n")

# Show top-ranked sentences
ranked = sorted(
    zip(example_sentences, importance_scores),
    key=lambda x: x[1],
    reverse=True
)

print("Top 3 most important sentences:")
for i, (s, score) in enumerate(ranked[:5], 1):
    print(f"{i}. ({score:.4f}) {s}")


In [ ]:
# ===============================
# Extractive summarization
# ===============================

def extractive_summary(sentences, scores, top_k=2):
    ranked = sorted(
        zip(sentences, scores),
        key=lambda x: x[1],
        reverse=True
    )
    return " ".join([s for s, _ in ranked[:top_k]])

extractive = extractive_summary(
    example_sentences,
    importance_scores,
    top_k=2
)

print("EXTRACTIVE SUMMARY\n")
print(extractive)


In [ ]:
# ===============================
# Abstractive summarization
# ===============================

t5_tokenizer = AutoTokenizer.from_pretrained("t5-small")
t5_model = AutoModelForSeq2SeqLM.from_pretrained("t5-small")

def abstractive_summary(text, max_len=25):
    input_text = "summarize: " + text
    inputs = t5_tokenizer(input_text, return_tensors="pt", truncation=True)
    output = t5_model.generate(**inputs, max_length=max_len)
    return t5_tokenizer.decode(output[0], skip_special_tokens=True)

abstractive = abstractive_summary(processed_reports[example_file])

print("ABSTRACTIVE SUMMARY\n")
print(abstractive)


In [ ]:
# ===============================
# Load ground-truth summaries
# ===============================

ground_truth = load_reports(os.path.join(BASE_DIR, "data/ground_truth"))

# Derive ground-truth filename
gt_file = example_file.replace(".txt", "_GT.txt")
gt = ground_truth.get(gt_file, None)

print("GROUND TRUTH SUMMARY (LLM)\n")
print(gt if gt else f"No ground truth found for {gt_file}")


In [ ]:
# ===============================
# Evaluation (multiple similarity metrics)
# ===============================

from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances
import numpy as np

def evaluate_summary_all(pred, ref):
    """
    Inputs:
        pred: str (generated summary)
        ref:  str (ground-truth summary)

    Outputs:
        dict with similarity scores
    """
    emb_pred = get_sentence_embedding(pred)
    emb_ref = get_sentence_embedding(ref)

    # 1. Cosine similarity
    cosine_sim = cosine_similarity(emb_pred, emb_ref)[0][0]

    # 2. Euclidean distance -> similarity
    euclid_dist = euclidean_distances(emb_pred, emb_ref)[0][0]
    euclid_sim = 1 / (1 + euclid_dist)

    # 3. Manhattan distance -> similarity
    manhattan_dist = manhattan_distances(emb_pred, emb_ref)[0][0]
    manhattan_sim = 1 / (1 + manhattan_dist)

    return {
        "cosine_similarity": cosine_sim,
        "euclidean_similarity": euclid_sim,
        "manhattan_similarity": manhattan_sim
    }

# ---- Run evaluation ----
if gt:
    extractive_scores = evaluate_summary_all(extractive, gt)
    abstractive_scores = evaluate_summary_all(abstractive, gt)

    print("Semantic similarity to ground truth (embedding-based)\n")

    print("EXTRACTIVE SUMMARY")
    for k, v in extractive_scores.items():
        print(f"{k}: {v:.4f}")

    print("\nABSTRACTIVE SUMMARY")
    for k, v in abstractive_scores.items():
        print(f"{k}: {v:.4f}")
